# derived_8.4-hybrid-lstm-1.6 — V21 BiLSTM+Attn Hidden-Size Sweep (no PCA)

In `derived_8.4-hybrid-lstm-1.5` (H=80), the raw LSTM representations (160-dim ctx, 80-dim head_hidden, 80-dim pre-ReLU) diluted the 54 tabular features in the hybrid XGBoost models, so PCA compression was needed to restore the tabular share. This experiment sweeps the LSTM hidden size over H ∈ {40, 20, 16, 8, 4} (1 seed each, seq_len=30) and evaluates the hybrid models on **raw (non-PCA)** representations to see whether a naturally compact hidden context removes the need for PCA. Leaderboard rows from `derived_8.4-hybrid-lstm-1.5` (H=80 ± PCA) are appended as `[1.5]` references.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Handle both: run from notebooks/ dir and run from the experiment dir itself
_cwd = Path.cwd().resolve()
if (_cwd / "experiment" / "derived_8.4-hybrid-lstm-1.6").is_dir():
    NOTEBOOK_DIR = _cwd / "experiment" / "derived_8.4-hybrid-lstm-1.6"
else:
    NOTEBOOK_DIR = _cwd
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from lstm.train import (
    train_lstm_for_hidden, ALL_FEATURES, SEQ_LEN, HIDDEN_SIZES,
)
from eval_hybrid.data import load_hybrid_experiment_data
from eval_hybrid.evaluator import HybridStrategyEvaluator
from eval_hybrid.shap_analysis import run_full_shap_analysis
from run_eval import (
    compute_c1_gain_additions, load_reference_1_5,
    build_variants, _model_name, _candidate_id,
)

with open(NOTEBOOK_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
MODELS_DIR = NOTEBOOK_DIR / "models"
DATA_DIR = PROJECT_ROOT / config.get("data_dir", "data/splits/derived_8.4")
hidden_sizes = [int(h) for h in config["hidden_sizes"]]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete. Hidden sizes:", hidden_sizes)

Setup complete. Hidden sizes: [40, 20, 16, 8, 4]


## Phase 1-3: V21 Training + Raw Representation Extraction (per hidden size)

For each hidden size H in {40, 20, 16, 8, 4}: train the V21 BiLSTM+Attn (1 seed), evaluate it standalone, and extract raw ctx (2H-dim) / head_hidden (H-dim) / head_pre_relu (H-dim) vectors into `artifacts/h{H}/`. If those files already exist, the training step is skipped.

In [2]:
lstm_metrics_map = {}
for h in hidden_sizes:
    h_dir = ARTIFACTS_DIR / f"h{h}"
    raw_names = ("ctx", "head_hidden", "head_pre_relu")
    has_raw = all((h_dir / f"{n}_{s}.npy").exists()
                  for n in raw_names for s in ("train", "val", "test"))
    has_metrics = (h_dir / "lstm_metrics.json").exists()
    if has_raw and has_metrics:
        print(f"[H{h}] Found existing representations + metrics. Skipping training.")
        with open(h_dir / "lstm_metrics.json") as f:
            lstm_metrics_map[h] = json.load(f)
        continue
    h_dir.mkdir(parents=True, exist_ok=True)
    best_seed_info, lstm_metrics = train_lstm_for_hidden(h, DATA_DIR, h_dir, MODELS_DIR)
    lstm_metrics_map[h] = lstm_metrics
    print(f"[H{h}] Best seed: {best_seed_info['seed']} (val_rmse={best_seed_info['val_rmse']:.5f})")

print("\nLSTM-only test metrics per hidden size:")
for h in hidden_sizes:
    t = lstm_metrics_map[h]["test"]
    print(f"  H{h}: R2={t['r2']:.4f} RMSE={t['rmse']:.5f} MAE={t['mae']:.5f}")

Phase 1: V21 LSTM Training (H=40, 1 seed)


[V21 H40] 58 features, seq_len=30, 1 seeds


[lstm dataset] train=(9803, 30, 58) val=(4805, 30, 58) test=(6620, 30, 58) (stride=1)


  [H40 Seed42 Ep  1] loss=0.00538 val_rmse=0.07230 best=0.07230 (ep1) lr=1.00e-03


  [H40 Seed42 Ep 20] loss=0.00057 val_rmse=0.05608 best=0.05543 (ep18) lr=5.00e-04


...

[lstm dataset] train=(9803, 30, 58) val=(4805, 30, 58) test=(6620, 30, 58) (stride=1)


  [ctx] train=(9803, 8) val=(4805, 8) test=(6620, 8)


  [hh] train=(9803, 4) val=(4805, 4) test=(6620, 4)


  [hp] train=(9803, 4) val=(4805, 4) test=(6620, 4)


[H4] Best seed: 42 (val_rmse=0.05965)

LSTM-only test metrics per hidden size:
  H40: R2=0.5523 RMSE=0.06816 MAE=0.05454
  H20: R2=0.6931 RMSE=0.05643 MAE=0.04398
  H16: R2=0.7032 RMSE=0.05550 MAE=0.04335
  H8: R2=0.6963 RMSE=0.05614 MAE=0.04558
  H4: R2=0.5263 RMSE=0.07011 MAE=0.05461


## Phase 4-5: XGBoost Tabular Baselines + Hybrid Models

Train the 2 tabular baselines (Global Single, Clustering_V0_Full_k2) and, for every (hidden size, representation) combination, the hybrid `[tabular + raw repr]` models under both strategies. No PCA is applied anywhere.

In [3]:
c1_additions = compute_c1_gain_additions(config)
backbone_54 = config["shared_backbone_54"]

data_base = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type="ctx", hidden_size=hidden_sizes[0])
eval_global = HybridStrategyEvaluator(data_base, config, "Global_Single", models_dir=MODELS_DIR)
eval_v0 = HybridStrategyEvaluator(data_base, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)

results = {}
results["Global Single (54 Backbone)"] = eval_global.fit_and_evaluate(
    model_name="Global Single (54 Backbone)", candidate_id="Global_Single_54_Backbone",
    global_features=backbone_54,
)
results["Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)"] = eval_v0.fit_and_evaluate(
    model_name="Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)",
    candidate_id="Clustering_V0_k2_54_Backbone", global_features=backbone_54,
    cluster_additions={"0": [], "1": c1_additions},
)

for h, repr_type in build_variants(hidden_sizes):
    data_v = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=repr_type, hidden_size=h)
    ev_g = HybridStrategyEvaluator(data_v, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(data_v, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    results[_model_name(h, repr_type, "Global_Single")] = ev_g.fit_and_evaluate(
        model_name=_model_name(h, repr_type, "Global_Single"),
        candidate_id=_candidate_id(h, repr_type, "Global_Single"),
        global_features=data_v.hybrid_features,
    )
    results[_model_name(h, repr_type, "Clustering_V0_Full_k2")] = ev_c.fit_and_evaluate(
        model_name=_model_name(h, repr_type, "Clustering_V0_Full_k2"),
        candidate_id=_candidate_id(h, repr_type, "Clustering_V0_Full_k2"),
        global_features=data_v.hybrid_features,
        cluster_additions={"0": [], "1": c1_additions},
    )

print(f"Trained {len(results)} XGBoost models.")

/scratch/group/p.cis250607.000/MDR-Project/notebooks/.venv/lib64/python3.12/site-packages/xgboost/core.py:751: UserWarning: [20:36:09] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Trained 32 XGBoost models.


## Phase 6: SHAP Feature Importance Analysis

Accelerated SHAP using XGBoost native `pred_contribs` (C++/CUDA tree traversal), computed for all trained hybrid + baseline models.

In [4]:
eval_map = {}
for h, repr_type in build_variants(hidden_sizes):
    dv = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=repr_type, hidden_size=h)
    ev_g = HybridStrategyEvaluator(dv, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(dv, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    eval_map[_model_name(h, repr_type, "Global_Single")] = ev_g
    eval_map[_model_name(h, repr_type, "Clustering_V0_Full_k2")] = ev_c

eval_map["Global Single (54 Backbone)"] = eval_global
eval_map["Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)"] = eval_v0

shap_info = run_full_shap_analysis(eval_map, results, ARTIFACTS_DIR)
print(f"SHAP computed for {len(shap_info['shap_results'])} models.")

Executing Accelerated SHAP Feature Importance Analysis


[SHAP] Computing SHAP values for: Global Single (54 Backbone)...


  -> Done in 1.407s (pred_contribs)


[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)...


  -> Done in 1.032s (pred_contribs)


...

[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10 + 4 Pre-ReLU [H4])...


  -> Done in 1.001s (pred_contribs)


[SHAP] Saved summary table to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.6/artifacts/shap_importance_summary.csv


[SHAP] Saved visualization plot to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.6/artifacts/shap_summary_plots.png


SHAP computed for 32 models.


## Results: Leaderboard

Combined leaderboard: this experiment's models (raw reps, H ∈ {40, 20, 16, 8, 4}) plus the `[1.5]` reference rows (H=80, with and without PCA).

In [5]:
records = [r.as_record() for r in results.values()]
for h in hidden_sizes:
    lstm_test = lstm_metrics_map[h]["test"]
    records.append({
        "model_name": f"BiLSTM+Attn H{h} (LSTM-only, V21)",
        "pooled_r2": lstm_test["r2"],
        "pooled_rmse": lstm_test["rmse"],
        "pooled_ubrmse": lstm_test["ubrmse"],
        "pooled_bias": lstm_test["bias"],
        "pooled_mae": lstm_test["mae"],
        "pooled_pearson": float("nan"),
        "year_2023_r2": float("nan"), "year_2024_r2": float("nan"), "year_2025_r2": float("nan"),
    })
df = pd.DataFrame(records)
df_ref = load_reference_1_5()
if not df_ref.empty:
    df = pd.concat([df, df_ref], ignore_index=True)
df = df.sort_values("pooled_r2", ascending=False).reset_index(drop=True)
display_cols = ["model_name", "pooled_r2", "pooled_rmse", "pooled_mae", "pooled_pearson"]
display(df[display_cols].round(4))

[Ref] Loaded 33 reference rows from derived_8.4-hybrid-lstm-1.5.


,model_name,pooled_r2,pooled_rmse,pooled_mae,pooled_pearson
0,"Clustering_V0_Full_k2 (54 Backbone, c0=0, c1=10)",0.8150,0.0438,0.0337,0.9056
1,"[1.5] Clustering_V0_Full_k2 (54 Backbone, c0=0...",0.8150,0.0438,0.0337,0.9056
2,[1.5] Global Single (54 Backbone),0.7792,0.0479,0.0371,0.8894
3,Global Single (54 Backbone),0.7792,0.0479,0.0371,0.8894
4,"[1.5] Clustering_V0_Full_k2 (54 Backbone, c0=0...",0.7667,0.0492,0.0384,0.8966
...,...,...,...,...,...
65,Global Single (54 Backbone + 40 Pre-ReLU [H40]),0.6262,0.0623,0.0494,0.8750
66,Global Single (54 Backbone + 80 CTX [H40]),0.6110,0.0635,0.0509,0.8731
67,"[1.5] BiLSTM+Attn (LSTM-only, V21)",0.5827,0.0658,0.0525,NaN
68,"BiLSTM+Attn H40 (LSTM-only, V21)",0.5523,0.0682,0.0545,NaN


## Save Artifacts & Generate README

Persist the leaderboard CSV and metrics JSON, then regenerate `README.md` from the executed outputs.

In [6]:
df.to_csv(ARTIFACTS_DIR / "summary_records.csv", index=False)
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump({k: r.as_record() for k, r in results.items()}, f, indent=2)

from run_eval import generate_readme
df_regime = pd.DataFrame()
generate_readme(df, df_regime, shap_info)
print("README generated.")


[Generated] /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.6/README.md


README generated.
